# 10 - Input-Type Margin Check + Live Recall@30/MRR Sanity Check

**Scope note:** this is the minimal, targeted verification called for in
`EMBEDDING_INPUT_TYPE_ASYMMETRY.md` Sec 2.5 -- NOT the full retrieval telemetry +
recall@k harness (that is separate, larger, ongoing work). This notebook answers
two narrow questions cheaply:

- **Part A (offline, ~$0.0001):** does embedding a query with `input_type=search_query`
  separate the correct evidence sentence from a distractor better than the old,
  buggy `search_document` role did? (Sec 2.5, steps 1-5 of the asymmetry doc.)
- **Part B (live, hits the real S3 Vectors index):** using the actual production
  path (`EntityAdapter` + `QueryEmbedderV2`, the code that now defaults to
  `search_query`), what is recall@30 / MRR@30 on the 10 hand-written P3.v3
  gold questions -- the trusted subset per `RETRIEVAL_IMPROVEMENT_STUDY.md` Sec 7.4
  (P3.v2 is excluded here: its evidence was selected by regex keyword match, which
  is circular for evaluating a retriever).

Both parts run against the index populated in this session (614,647 vectors,
`finrag-sentence-fact-embed-1024d`).

In [1]:
import sys
from pathlib import Path

for p in [Path.cwd()] + list(Path.cwd().parents):
    if p.name == "ModelPipeline":
        MODEL_ROOT = p
        break
if str(MODEL_ROOT) not in sys.path:
    sys.path.insert(0, str(MODEL_ROOT))

import json
import numpy as np
import polars as pl
import boto3

from finrag_ml_tg1.loaders.ml_config_loader import MLConfig

config = MLConfig()

bedrock_cfg = config.cfg["embedding"]["bedrock"]
BEDROCK_MODEL_ID = bedrock_cfg["models"][bedrock_cfg["default_model"]]["model_id"]
DIM = config.s3vectors_dimensions("cohere_1024d")
VECTOR_BUCKET = config.cfg["retrieval"]["vector_bucket"]
INDEX_NAME = config.cfg["retrieval"]["index_name"]

print(f"Model:  {BEDROCK_MODEL_ID}  (dim={DIM})")
print(f"Index:  {VECTOR_BUCKET} / {INDEX_NAME}")
print(f"Config-resolved input types: document={config.cfg['embedding']['spec']['input_type_document']!r}, "
      f"query={config.cfg['embedding']['spec']['input_type_query']!r}")

[DEBUG] ✓ Found ModelPipeline via file path: /Users/joel/MJS_ROOT/MJS_STUDY/NEU Sem5 - MLOPS/MLOps Project/FinSights/ModelPipeline
[DEBUG] ✓ AWS credentials loaded from aws_credentials.env
Model:  cohere.embed-v4:0  (dim=1024)
Index:  finrag-embeddings-s3vectors / finrag-sentence-fact-embed-1024d
Config-resolved input types: document='search_document', query='search_query'


## Part A - Offline margin check (no index calls)

For each of the 10 P3.v3 questions: embed the question twice (once
`search_query`, once the old buggy `search_document`), embed the correct
evidence sentence and one distractor (another question's evidence, a genuinely
different company/topic) with `search_document`, and compare the
correct-vs-distractor cosine margin under each role.

**Prediction being tested** (falsifiable, written down before running, per
Sec 2.5): the `search_query` margin should be larger than the `search_document`
margin for most questions. If it isn't, the input_type hypothesis is wrong.

In [2]:
GOLD_PATH = MODEL_ROOT.parent / "MLFlow_POC" / "data" / "p3_gold_test_suite_31q.json"
gold = json.loads(GOLD_PATH.read_text())
p3v3 = [q for q in gold if q.get("gold_version") == "P3.v3"]
assert len(p3v3) == 10, f"expected 10 P3.v3 questions, got {len(p3v3)}"
print(f"Loaded {len(p3v3)} P3.v3 (trusted, hand-written) gold questions")

first_evidence_ids = [q["evidence_sentence_ids"][0] for q in p3v3]

meta_path = (config.model_root / "finrag_ml_tg1" / "data_cache" / "meta_embeds"
             / "finrag_fact_sentences_meta_embeds.parquet")
sentence_lookup = (
    pl.scan_parquet(meta_path)
      .filter(pl.col("sentenceID").is_in(first_evidence_ids))
      .select(["sentenceID", "sentence"])
      .collect()
)
text_by_id = dict(zip(sentence_lookup["sentenceID"].to_list(), sentence_lookup["sentence"].to_list()))
missing = [sid for sid in first_evidence_ids if sid not in text_by_id]
assert not missing, f"missing evidence text for: {missing}"
print(f"Resolved evidence sentence text for all {len(text_by_id)} anchors")

Loaded 10 P3.v3 (trusted, hand-written) gold questions
Resolved evidence sentence text for all 10 anchors


In [3]:
bedrock = config.get_bedrock_client()

def embed_direct(text: str, input_type: str) -> np.ndarray:
    """Direct Bedrock invoke, bypassing QueryEmbedderV2 so input_type can be
    varied explicitly for the A/B (QueryEmbedderV2 now always sends search_query,
    which is correct for production but means we can't force the old role through it)."""
    body = json.dumps({
        "texts": [text],
        "input_type": input_type,
        "embedding_types": ["float"],
        "output_dimension": DIM,
        "max_tokens": 128000,
        "truncate": "RIGHT",
    })
    resp = bedrock.invoke_model(
        modelId=BEDROCK_MODEL_ID, contentType="application/json",
        accept="application/json", body=body,
    )
    parsed = json.loads(resp["body"].read())
    return np.asarray(parsed["embeddings"]["float"][0], dtype=np.float32)

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

rows = []
for i, q in enumerate(p3v3):
    qtext = q["question_text"]
    evi_text = text_by_id[q["evidence_sentence_ids"][0]]
    dist_text = text_by_id[p3v3[(i + 1) % len(p3v3)]["evidence_sentence_ids"][0]]

    u_query = embed_direct(qtext, "search_query")
    u_doc   = embed_direct(qtext, "search_document")
    v_evi   = embed_direct(evi_text, "search_document")
    v_dist  = embed_direct(dist_text, "search_document")

    cos_query_evi  = cosine(u_query, v_evi)
    cos_doc_evi    = cosine(u_doc, v_evi)
    cos_query_dist = cosine(u_query, v_dist)
    cos_doc_dist   = cosine(u_doc, v_dist)

    rows.append({
        "question_id": q["question_id"],
        "cos_query_evi": round(cos_query_evi, 4),
        "cos_doc_evi": round(cos_doc_evi, 4),
        "cos_query_dist": round(cos_query_dist, 4),
        "cos_doc_dist": round(cos_doc_dist, 4),
        "margin_query": round(cos_query_evi - cos_query_dist, 4),
        "margin_doc": round(cos_doc_evi - cos_doc_dist, 4),
    })

margins = pl.DataFrame(rows)
print(margins)

n = len(margins)
n_query_wins = int((margins["margin_query"] > margins["margin_doc"]).sum())
print(f"\nquestions where search_query margin > search_document margin: {n_query_wins}/{n}")
print(f"mean margin (search_query):    {margins['margin_query'].mean():.4f}")
print(f"mean margin (search_document): {margins['margin_doc'].mean():.4f}")
print(f"mean cos(query-role, evidence):    {margins['cos_query_evi'].mean():.4f}")
print(f"mean cos(document-role, evidence): {margins['cos_doc_evi'].mean():.4f}")

shape: (10, 7)
┌─────────────┬──────────────┬─────────────┬──────────────┬─────────────┬─────────────┬────────────┐
│ question_id ┆ cos_query_ev ┆ cos_doc_evi ┆ cos_query_di ┆ cos_doc_dis ┆ margin_quer ┆ margin_doc │
│ ---         ┆ i            ┆ ---         ┆ st           ┆ t           ┆ y           ┆ ---        │
│ str         ┆ ---          ┆ f64         ┆ ---          ┆ ---         ┆ ---         ┆ f64        │
│             ┆ f64          ┆             ┆ f64          ┆ f64         ┆ f64         ┆            │
╞═════════════╪══════════════╪═════════════╪══════════════╪═════════════╪═════════════╪════════════╡
│ P3V3-Q001   ┆ 0.3319       ┆ 0.3486      ┆ 0.1527       ┆ 0.1792      ┆ 0.1793      ┆ 0.1694     │
│ P3V3-Q002   ┆ 0.3837       ┆ 0.3872      ┆ 0.0813       ┆ 0.0886      ┆ 0.3024      ┆ 0.2986     │
│ P3V3-Q003   ┆ 0.2388       ┆ 0.2367      ┆ 0.1265       ┆ 0.1519      ┆ 0.1123      ┆ 0.0848     │
│ P3V3-Q004   ┆ 0.3216       ┆ 0.3312      ┆ 0.3466       ┆ 0.3447      ┆ -0

## Part B - Live recall@30 / MRR mini-check (real S3 Vectors index)

Uses the actual production construction (`EntityAdapter.extract` ->
`QueryEmbedderV2.embed_query`, the same call path as `run_supply_line_2_rag`),
open regime (no metadata filter -- the harder, more honest test), `topK=30`
matching the `recall@30` convention used in `RETRIEVAL_IMPROVEMENT_STUDY.md` Sec 7.
Uses **all** evidence_sentence_ids per question (not just the first), so
multi-evidence questions get a real fractional recall score.

In [4]:
from finrag_ml_tg1.loaders.data_loader_factory import create_data_loader
from finrag_ml_tg1.rag_modules_src.entity_adapter.entity_adapter import EntityAdapter
from finrag_ml_tg1.rag_modules_src.utilities.query_embedder_v2 import (
    QueryEmbedderV2, EmbeddingRuntimeConfig,
)

data_loader = create_data_loader(config)
adapter = EntityAdapter(data_loader=data_loader)
runtime_cfg = EmbeddingRuntimeConfig.from_ml_config(config.cfg["embedding"])
embedder = QueryEmbedderV2(runtime_cfg, boto_client=config.get_bedrock_client())

print(f"Live query embedder input_type = {runtime_cfg.input_type!r}")
assert runtime_cfg.input_type == "search_query", "REGRESSION: query embedder is not using search_query"

s3v = boto3.client(
    "s3vectors", region_name=config.region,
    aws_access_key_id=config.aws_access_key,
    aws_secret_access_key=config.aws_secret_key,
)

TOPK = 30

def query_topk(qvec, topk=TOPK):
    resp = s3v.query_vectors(
        vectorBucketName=VECTOR_BUCKET,
        indexName=INDEX_NAME,
        queryVector={"float32": list(qvec)},
        topK=topk,
        returnMetadata=True,
        returnDistance=True,
    )
    hits = resp.get("vectors", [])
    return [h.get("metadata", {}).get("sentenceID") for h in hits]

def first_hit_rank(retrieved_ids, gold_ids):
    gold_set = set(gold_ids)
    for i, sid in enumerate(retrieved_ids, start=1):
        if sid in gold_set:
            return i
    return None

recall_rows = []
for q in p3v3:
    try:
        entities = adapter.extract(q["question_text"])
        qvec = embedder.embed_query(q["question_text"], entities)
    except Exception as e:
        print(f"  [SKIP] {q['question_id']}: {type(e).__name__}: {e}")
        continue

    retrieved = query_topk(qvec, topk=TOPK)
    gold_ids = q["evidence_sentence_ids"]
    n_hits = sum(1 for sid in gold_ids if sid in retrieved)
    rank = first_hit_rank(retrieved, gold_ids)

    recall_rows.append({
        "question_id": q["question_id"],
        "n_evidence": len(gold_ids),
        "n_hits": n_hits,
        "recall_at_30": round(n_hits / len(gold_ids), 3),
        "first_hit_rank": rank,
        "reciprocal_rank": round(1.0 / rank, 4) if rank else 0.0,
    })

recall_df = pl.DataFrame(recall_rows)
print(recall_df)
print(f"\nMean recall@{TOPK}: {recall_df['recall_at_30'].mean():.3f}")
print(f"MRR@{TOPK}: {recall_df['reciprocal_rank'].mean():.3f}")
print(f"Questions with >=1 evidence hit: {(recall_df['n_hits'] > 0).sum()}/{len(recall_df)}")

[DEBUG] ✓ Local dev detected → LOCAL_CACHE mode
Live query embedder input_type = 'search_query'


shape: (10, 6)
┌─────────────┬────────────┬────────┬──────────────┬────────────────┬─────────────────┐
│ question_id ┆ n_evidence ┆ n_hits ┆ recall_at_30 ┆ first_hit_rank ┆ reciprocal_rank │
│ ---         ┆ ---        ┆ ---    ┆ ---          ┆ ---            ┆ ---             │
│ str         ┆ i64        ┆ i64    ┆ f64          ┆ i64            ┆ f64             │
╞═════════════╪════════════╪════════╪══════════════╪════════════════╪═════════════════╡
│ P3V3-Q001   ┆ 4          ┆ 0      ┆ 0.0          ┆ null           ┆ 0.0             │
│ P3V3-Q002   ┆ 4          ┆ 0      ┆ 0.0          ┆ null           ┆ 0.0             │
│ P3V3-Q003   ┆ 3          ┆ 1      ┆ 0.333        ┆ 1              ┆ 1.0             │
│ P3V3-Q004   ┆ 3          ┆ 0      ┆ 0.0          ┆ null           ┆ 0.0             │
│ P3V3-Q005   ┆ 4          ┆ 0      ┆ 0.0          ┆ null           ┆ 0.0             │
│ P3V3-Q006   ┆ 2          ┆ 0      ┆ 0.0          ┆ null           ┆ 0.0             │
│ P3V3-Q007   ┆ 1

## Interpretation (fill in after running)

- If `margin_query` beats `margin_doc` on most of the 10 questions in Part A, that
  is direct evidence the input_type fix moves embeddings in the predicted
  direction -- separate from and prior to any index-level result.
- Part B's recall@30/MRR is a first real number for the *current*, fixed pipeline
  against the *current* index (614,647 vectors) -- there was no prior baseline to
  compare against on this account (old account's numbers are unrecoverable), so
  read this as a fresh floor, not an improvement delta.
- This is intentionally a 10-question, single-regime, single-run check. It does
  **not** replace the full #9 harness (all usable questions, filtered + open
  regimes, McNemar significance test, telemetry logging) -- see
  `RETRIEVAL_IMPROVEMENT_STUDY.md` Sec 7 for that scope.